In [ ]:
#@title Imports
import numpy as np
import copy
import cv2
import matplotlib.pyplot as plt
import tensorflow as tf
from keras.models import Model, load_model
import pickle
import glob
import numpy.ma as ma # For masked arrays
import imageio

#Metódy pre výpočet chybovosti.

calculateClassifErrors má na vstupe dve matice, predpokladá sa, že int, s tým, že obsahujú len 1 a 0.  Matice sú predikcia a ground truth: GT.
Na výstupe je matica Errors, ktorá má 0 na true negative, 2 na true positive, -1 na false negative a 1 na false positive.

In [ ]:
def calculateClassifErrors(predicted_classif,GT):
  # print('black: 0(TN), white: 2(TP), blue: -1(FN), red: 1(FP)')
  Errors = ((predicted_classif + GT) > 1)*2.0
  Errors = Errors + (predicted_classif - GT)
  return Errors

Na zobrazenie rsp. uloženie vizualizácie chýb ako obrázku slúžia tieto dve metódy:

In [ ]:
def showErrors(Errors):
  from matplotlib.figure import Figure
  from matplotlib.colors import LinearSegmentedColormap
  colors = [(0, 0, 1), (0, 0, 0), (1, 0, 0), (1, 1, 1)]  # Blue, Black, Red, White
  cmap_name = 'toShow'
  cmapToShow = LinearSegmentedColormap.from_list(cmap_name, colors, N=4)

  fig = Figure(figsize=(30, 20))
  # pic = fig.add_subplot(111)
  plt.imshow(Errors, cmap=cmapToShow)
  plt.savefig('classifVsGT.png', dpi=1200)
  plt.show()

  print('black: 0(TN), white: 2(TP), blue: -1(FN), red: 1(FP)')

def saveErrorsAsPNG(outfilePath, Errors):
  colors = [(0, 0, 1), (0, 0, 0), (1, 0, 0), (1, 1, 1)]  # Blue, Black, Red, White
  # print('black: 0(TN), white: 2(TP), blue: -1(FN), red: 1(FP)')
  img = np.zeros((Errors.shape[0],Errors.shape[1],3),dtype = np.dtype('uint8'))
  img[:,:,0] = (Errors == 2) + (Errors == 1)
  img[:,:,1] = (Errors == 2)
  img[:,:,2] = (Errors == 2) + (Errors == -1)
  img = img*255

  imageio.imwrite(outfilePath, img)

Na výpočet štatistických ukazovateľov z matice Error slúži táto metóda:
Pre detaily, čo je čo viď.: https://en.wikipedia.org/wiki/Confusion_matrix


In [ ]:
def calculateStatistics(Errors):
  # ('black: 0(TN), white: 2(TP), blue: -1(FN), red: 1(FP)')
  # reconstruct predicted and ground truth
  predicted = np.zeros(Errors.shape, dtype='uint8')
  GT = np.zeros(Errors.shape, dtype='uint8')
  predicted = predicted + (Errors==2) + (Errors==1)
  GT = GT + (Errors==2) + (Errors==-1)

  TN = np.sum(Errors==0)
  TP = np.sum(Errors==2)
  FN = np.sum(Errors==-1)
  FP = np.sum(Errors==1)

  P = TP + FN # Possitive examples
  N = TN + FP # Negative examples

  # ConfusionMatrix = np.array([[TP, FN],[FP, TN]])
  # ConfusionMatrixPercentual = ConfusionMatrix / (P + N)

  TPR = TP/(TP+FN) # Sensitivity, true possitive rate
  TNR = TN/(TN+FP) # Specificity, true negative rate


  BalancedAccuracy = (TPR + TNR)/2
  Accuracy = (TP + TN)/(P + N)

  MOR10R = countMisclassifiedOutOfRadius(Errors, 10)/(P+N)

  IoU_fore, IoU_bgr, IoU_ave  = intersectionOverUnion(predicted, GT)

  PPV = TP/(TP + FP) # Positive predictive value, precision
  NPV = TN/(TN + FN) # Negative predictive value

  F1 = 2*TP/(2*TP + FP +FN)

  TN = TN.astype('float64')
  TP = TP.astype('float64')
  FN = FN.astype('float64')
  FP = FP.astype('float64')
  MCC = (TP * TN - FP * FN)/np.sqrt(((TP+FP)*(TP+FN)*(TN+FP)*(TN+FN))) # Matthews correlation coef.


  Statistics = dict(
      # ConfusionMatrix = ConfusionMatrix,
      # ConfusionMatrixPercentual = ConfusionMatrixPercentual,
      BalancedAccuracy = BalancedAccuracy,
      Accuracy = Accuracy,
      TN = TN,
      TP = TP,
      FN = FN,
      FP = FP,
      TPR = TPR,
      TNR = TNR,
      PPV = PPV,
      NPV = NPV,
      F1 = F1,
      MOR10R = MOR10R,
      IoU_fore = IoU_fore,
      IoU_bgr = IoU_bgr,
      IoU_ave = IoU_ave,
      MCC = MCC
  )
  # print(ConfusionMatrix)
  # print(BalancedAccuracy)

  return Statistics

def countMisclassifiedOutOfRadius(errors, radius):
  # print('black: 0(TN), white: 2(TP), blue: -1(FN), red: 1(FP)')
  TP = errors == 2
  Misclassified = np.logical_or((errors == -1),(errors == 1))
  TP = extendToRadius(TP,radius)

  return np.sum(Misclassified) - np.sum(np.logical_and(Misclassified,TP))

def intersectionOverUnion(prediction, groundTruth):
  pr = prediction > 0
  gt = groundTruth > 0
  intersection = np.logical_and(pr, gt)
  union = np.logical_or(pr, gt)
  iou_fore = np.sum(intersection) / np.sum(union)
  # print('iou_fore = %s' % iou_fore)
  pr = prediction < 1
  gt = groundTruth < 1
  intersection = np.logical_and(pr, gt)
  union = np.logical_or(pr, gt)
  iou_bgr = np.sum(intersection) / np.sum(union)
  # print('iou_bgr = %s' % iou_bgr)
  iou_ave = (iou_fore + iou_bgr)/2
  # print('iou_ave = %s' % iou_ave)

  return iou_fore, iou_bgr, iou_ave

def extendToRadius(binary_img,radius):
  # shadowLength is in meters to all directions from an object
  img = binary_img.astype('uint8')
  kernel = np.ones((3,3), np.uint8)
  kernel[0,0] = 0
  kernel[0,2] = 0
  kernel[2,0] = 0
  kernel[2,2] = 0

  extended = cv2.dilate(img, kernel, iterations=radius)

  return extended == 1

ak je treba zrekonštruovať predikciu a GT z matice Errors

In [ ]:
def reconstructFromErrors(Errors):
  predicted = np.zeros(Errors.shape, dtype='uint8')
  GT = np.zeros(Errors.shape, dtype='uint8')
  predicted = predicted + (Errors==2) + (Errors==1)
  GT = GT + (Errors==2) + (Errors==-1)

  return predicted, GT

#Vyhodnotenie počtu zásahov objektov

valid_predicted_output je predikcia pre vyšetrovanú oblasť \\
valid_GT je ground truth pre vyšetrovanú oblasť \\
Errors je matica chýb, ako bolo definované vyššie \\
Toto sú funkcie, ktoré treba na analýzu zásahov. Nie sú práve optimalizované pre looting ale mali by poslúžiť aj bez modifikácie.

In [ ]:
def calculateAreas(labeled_img):
  no_of_objects = np.max(labeled_img)
  sizes = np.zeros((no_of_objects),dtype='uint32')
  for i in range(1,no_of_objects+1,1):
    sizes[i-1] = np.sum(labeled_img == i)

  return sizes

def hitOrMiss(labeled_img, errors):
  no_of_objects = np.max(labeled_img)
  # hits = np.zeros(no_of_objects,dtype='bool')
  hits = np.zeros(no_of_objects,dtype='uint16')
  predicted = np.zeros(errors.shape, dtype='bool')
  predicted = np.logical_or((errors==2),(errors==1))

  for i in range(1,no_of_objects+1,1):
    C = np.sum(np.logical_and((labeled_img == i),predicted)) # Count of overlapping pixels
    if C>1: # Criterium to consider it a hit 1 pixel overlap
      # hits[i-1] = True
      hits[i-1] = C

  # hitRatio = np.sum(hits)/hits.size
  return hits

def analyzeHits(hits_counts, sizes, sizeBreakpoints):

  print('Average number of hits: %.6f' % (np.sum(hits_counts)/hits_counts.size))

  hits = copy.deepcopy(hits_counts)
  hits = hits>0

  print('Average number of hits in hitted: %.6f' % (np.sum(hits_counts)/np.sum(hits)))

  indices = np.argsort(sizes)
  sizes = sizes[indices]
  hits = hits[indices]
  hitRatio = np.sum(hits)/hits.size
  print('Hit ratio: %.6f' % hitRatio)
  i = 0
  startIndex = 0
  endIndex = 0
  # endSize = sizeBreakpoints[i]
  for endSize in sizeBreakpoints:
    while sizes[endIndex]<=endSize:
      endIndex+=1
    hitRatioInterval= np.sum(hits[startIndex:endIndex])/hits[startIndex:endIndex].size
    print('Hit ratio between sizes %d and %d: %.6f' % (sizes[startIndex], sizes[endIndex], hitRatioInterval))
    startIndex = endIndex

  hitRatioInterval= np.sum(hits[startIndex:-1])/hits[startIndex:-1].size
  print('Hit ratio between sizes %d and %d: %.6f' % (sizes[startIndex], sizes[-1], hitRatioInterval))



Pre analýzu zásahov objektov treba použiť túto postupnosť.

In [ ]:
num_labels, labeled_img = cv2.connectedComponents((valid_GT>0).astype('uint8'))
print('No. of objects in the set: ' + str(num_labels))

No. of objects in the set: 1506


In [ ]:
# calculate objects
sizes = calculateAreas(labeled_img)
counts, bins = np.histogram(sizes, range(np.min(sizes),np.max(sizes)+1,1))
countsCumulative = np.cumsum(counts)
print('Smallest object: ' + str(np.min(sizes)))
print('Largest object: ' + str(np.max(sizes)))

Smallest object: 9
Largest object: 5510


In [ ]:
index = 0
while countsCumulative[index]<(np.max(countsCumulative)*0.5): # 50% objects
  index+=1
print(index)
index50 = index
index = 0
while countsCumulative[index]<(np.max(countsCumulative)*0.95): # 95% objects
  index+=1
print(index)
index95 = index


64
254


In [ ]:
hits = hitOrMiss(labeled_img, Errors)
analyzeHits(hits, sizes, [index50, index95])

Average number of hits: 19.409967
Average number of hits in hitted: 75.095116
Hit ratio: 0.258472
Hit ratio between sizes 9 and 65: 0.116719
Hit ratio between sizes 65 and 255: 0.338422
Hit ratio between sizes 255 and 5510: 0.571429


In [ ]:
# Pocitanie zasahov looting trenches
prediction = valid_predicted_output>0 # Mask RCNN
n_l, l_img = cv2.connectedComponents((prediction).astype('uint8'))
tp = (n_l-1)
print('No. of trenches predicted: ' + str(tp))
tcp = np.sum(hits>0)
print('No. of correctly predicted: ' + str(tcp))
print('No. of falsely predicted: ' + str(tp-tcp))

#Vyhodnotenie zásahov z PNG výstupu


In [ ]:
def getErrorsFromPNG(filepath):
  data = cv2.imread(filepath)
  data = data/255
  data = (data[:,:,0]*1 + data[:,:,1]*2 +data[:,:,2]*3)
  # data: 0 = TN, 1 = TP, 2 = FP, 3 = FN (zistit, preco su F naopak)
  data = np.zeros(data.shape,dtype = 'uint8') + (data == 6)*1 + (data == 3)*2 + (data == 1)*3
  # Errors: ('black: 0(TN), white: 2(TP), blue: -1(FN), red: 1(FP)')
  Errors = np.zeros(data.shape)
  Errors += (data==1)*2 + (data==2)*1 + (data==3)*-1

  return Errors

def targetHits(target, shots):
  num_labels, labeled_img = cv2.connectedComponents(target)

  no_of_objects = np.max(labeled_img)
  # hits = np.zeros(no_of_objects,dtype='bool')
  hits = np.zeros(no_of_objects,dtype='uint16')
  # predicted = np.zeros(errors.shape, dtype='bool')
  # predicted = np.logical_or((errors==2),(errors==1))
  predicted = shots

  for i in range(1,no_of_objects+1,1):
    C = np.sum(np.logical_and((labeled_img == i),predicted)) # Count of overlapping pixels
    if C>1: # Criterium to consider it a hit 1 pixel overlap
      # hits[i-1] = True
      hits[i-1] = C

  # hitRatio = np.sum(hits)/hits.size
  return hits


In [ ]:
# Structures
# Errors = getErrorsFromPNG('/content/drive/My Drive/Quatemala/Sets/UNet_Structures/Presentation/Test_IOU_based_box30_valid.png')
# Errors = getErrorsFromPNG('/content/drive/My Drive/Quatemala/Sets/UNet_Structures/Presentation/MRCNN_valid_01_my.png')
Errors = getErrorsFromPNG('/content/drive/My Drive/Quatemala/Sets/UNet_Structures/Te_opened_closed.png') # STructures, Unet

# Mounds
# Errors = getErrorsFromPNG('/content/drive/My Drive/Quatemala/Sets/UNet_Mounds/pics and results/Test/Test_Accuracy_based_valid.png')
# Errors = getErrorsFromPNG('/content/drive/My Drive/Quatemala/Sets/UNet_Mounds/pics and results/Test/Test_IoU_ave_based_valid.png')
# Errors = getErrorsFromPNG('/content/drive/My Drive/Quatemala/Sets/UNet_Mounds/pics and results/Test/Test_bf30_MOR10R_based_valid.png')
# Errors = getErrorsFromPNG('/content/drive/My Drive/Quatemala/Sets/UNet_Mounds/MRCNN/Test_MRCNN.png')

predicted, GT = reconstructFromErrors(Errors)

In [ ]:
num_labels, labeled_img = cv2.connectedComponents(GT)
# calculate objects
sizes = calculateAreas(labeled_img)
counts, bins = np.histogram(sizes, range(np.min(sizes),np.max(sizes)+1,1))
countsCumulative = np.cumsum(counts)
print('Smallest object: ' + str(np.min(sizes)))
print('Largest object: ' + str(np.max(sizes)))

index = 0
while countsCumulative[index]<(np.max(countsCumulative)*0.5): # 50% objects
  index+=1
#print(index)
index50 = index
index = 0
while countsCumulative[index]<(np.max(countsCumulative)*0.95): # 95% objects
  index+=1
#print(index)
index95 = index

hits = hitOrMiss(labeled_img, Errors)
analyzeHits(hits, sizes, [index50, index95])

n_l, l_img = cv2.connectedComponents(predicted)
tp = (n_l-1)
print('No. of real objects: ' + str(num_labels))
tcp = np.sum(hits>0)
print('From the real objects correctly predicted: ' + str(tcp))

print('Total no. of objects predicted (may occur overlap): ' + str(tp))
h = targetHits(target = predicted, shots = GT)
fp = np.sum(h<1)
print('No. of hitting objects: ' + str(tp-fp))
print('No. of missing objects: ' + str(fp))



Smallest object: 13
Largest object: 43909
Average number of hits: 405.434615
Average number of hits in hitted: 666.116904
Hit ratio: 0.608654
Hit ratio between sizes 13 and 213: 0.448898
Hit ratio between sizes 213 and 2183: 0.731006
Hit ratio between sizes 2183 and 43909: 0.981132
No. of real objects: 1041
From the real objects correctly predicted: 633
Total no. of objects predicted (may occur overlap): 682
No. of hitting objects: 516
No. of missing objects: 166


In [ ]:
# Test Posefkovho vystupu

path = "/content/drive/MyDrive/Quatemala/Sets/Posefko/error.png"
Errors = getErrorsFromPNG(path)
Statistics = calculateStatistics(Errors)




In [ ]:
print(Statistics)

{'BalancedAccuracy': 0.6720054559698404, 'Accuracy': 0.9968884848791593, 'TN': 75901486.0, 'TP': 87223.0, 'FN': 165632.0, 'FP': 71546.0, 'TPR': 0.34495264084158905, 'TNR': 0.9990582710980918, 'PPV': 0.5493704690462244, 'NPV': 0.9978225545497859, 'F1': 0.42379938973432063, 'MOR10R': 0.001947212500131353, 'IoU_fore': 0.2688740170344728, 'IoU_bgr': 0.9968849203868353, 'IoU_ave': 0.632879468710654, 'MCC': 0.43386676647825284}
